# Gravity Student Lab (Minimal View)

No coding steps are required in this notebook.

1. Choose model settings in **Gravity Spec Controls**.
2. Run **Run Gravity Regression**.
3. Read the coefficient table and diagnostics.

📖 **[Student Quick Guide (full instructions)](https://github.com/DTEcon/Teaching_International_Trade_PUBLIC/blob/main/INSTRUCTIONS_GRAVITY_LAB.html)**

**How to use this (quick guide)**
1. Set `preset_spec` first.
   - If you choose `Custom`, you manually choose FE inclusion (`include_fe`), covariates, and exporter/importer sample.
   - If you choose a non-`Custom` preset, the full specification is auto-set.
   - `Canonical FE (OLS)` and `Canonical FE (PPML)` reproduce the estimates shown in Lecture 7, slide 26.
2. Click `Runtime -> Run all`.
3. Read the output table and diagnostics in **Run Gravity Regression**.
4. Optional: click **Show code** in **Gravity Spec Controls** and **Run Gravity Regression** to see the underlying data and regression code.

**Save note (optional)**
- To keep your own changes (settings or code), use `File -> Save a copy in Drive`.
- You can also download a local copy via `File -> Download -> Download .ipynb`.

In [ ]:
# @title Gravity Spec Controls
# @markdown 📖 **[Student Quick Guide (full instructions)](https://github.com/DTEcon/Teaching_International_Trade_PUBLIC/blob/main/INSTRUCTIONS_GRAVITY_LAB.html)**
preset_spec = "Canonical FE (OLS)" # @param ["Naive GDP gravity (no FE)", "Cost-proxy gravity (no FE)", "Canonical FE (OLS)", "Canonical FE (PPML)", "Custom"]
estimator = "OLS" # @param ["OLS", "PPML"]
include_fe = True # @param {type:"boolean"}
exporters = "ALL" # @param {type:"string"}
importers = "ALL" # @param {type:"string"}

# @markdown ### Bilateral covariates
cov_ln_dist = True # @param {type:"boolean"}
cov_contig = True # @param {type:"boolean"}
cov_comlang_off = True # @param {type:"boolean"}
cov_rta = True # @param {type:"boolean"}
cov_dist = False # @param {type:"boolean"}
cov_comcol = False # @param {type:"boolean"}
cov_comleg_posttrans = False # @param {type:"boolean"}

# @markdown ### Exporter-only covariates
cov_ln_gdp_o = False # @param {type:"boolean"}
cov_gdp_o = False # @param {type:"boolean"}
cov_ln_pop_o = False # @param {type:"boolean"}
cov_pop_o = False # @param {type:"boolean"}
cov_ln_gdpcap_o = False # @param {type:"boolean"}
cov_gdpcap_o = False # @param {type:"boolean"}
cov_wto_o = False # @param {type:"boolean"}
cov_eu_o = False # @param {type:"boolean"}

# @markdown ### Importer-only covariates
cov_ln_gdp_d = False # @param {type:"boolean"}
cov_gdp_d = False # @param {type:"boolean"}
cov_ln_pop_d = False # @param {type:"boolean"}
cov_pop_d = False # @param {type:"boolean"}
cov_ln_gdpcap_d = False # @param {type:"boolean"}
cov_gdpcap_d = False # @param {type:"boolean"}
cov_wto_d = False # @param {type:"boolean"}
cov_eu_d = False # @param {type:"boolean"}

BILATERAL_COVARIATE_VARS = [
    ("ln_dist", "cov_ln_dist"),
    ("contig", "cov_contig"),
    ("comlang_off", "cov_comlang_off"),
    ("rta", "cov_rta"),
    ("dist", "cov_dist"),
    ("comcol", "cov_comcol"),
    ("comleg_posttrans", "cov_comleg_posttrans"),
]

EXPORTER_COVARIATE_VARS = [
    ("ln_gdp_o", "cov_ln_gdp_o"),
    ("gdp_o", "cov_gdp_o"),
    ("ln_pop_o", "cov_ln_pop_o"),
    ("pop_o", "cov_pop_o"),
    ("ln_gdpcap_o", "cov_ln_gdpcap_o"),
    ("gdpcap_o", "cov_gdpcap_o"),
    ("wto_o", "cov_wto_o"),
    ("eu_o", "cov_eu_o"),
]

IMPORTER_COVARIATE_VARS = [
    ("ln_gdp_d", "cov_ln_gdp_d"),
    ("gdp_d", "cov_gdp_d"),
    ("ln_pop_d", "cov_ln_pop_d"),
    ("pop_d", "cov_pop_d"),
    ("ln_gdpcap_d", "cov_ln_gdpcap_d"),
    ("gdpcap_d", "cov_gdpcap_d"),
    ("wto_d", "cov_wto_d"),
    ("eu_d", "cov_eu_d"),
]

COVARIATE_CHECKBOX_VARS = dict(BILATERAL_COVARIATE_VARS + EXPORTER_COVARIATE_VARS + IMPORTER_COVARIATE_VARS)

PRESETS = {
    "Naive GDP gravity (no FE)": {
        "estimator": "OLS",
        "include_fe": False,
        "covariates": ["ln_dist", "ln_gdp_o", "ln_gdp_d"],
    },
    "Cost-proxy gravity (no FE)": {
        "estimator": "OLS",
        "include_fe": False,
        "covariates": ["ln_dist", "ln_gdp_o", "ln_gdp_d", "contig", "comlang_off", "rta"],
    },
    "Canonical FE (OLS)": {
        "estimator": "OLS",
        "include_fe": True,
        "covariates": ["ln_dist", "contig", "comlang_off", "rta"],
    },
    "Canonical FE (PPML)": {
        "estimator": "PPML",
        "include_fe": True,
        "covariates": ["ln_dist", "contig", "comlang_off", "rta"],
    },
}


def _apply_preset_covariates(covariates: list[str]) -> None:
    selected = set(covariates)
    for covariate, var_name in COVARIATE_CHECKBOX_VARS.items():
        globals()[var_name] = covariate in selected


if preset_spec != "Custom":
    spec = PRESETS[preset_spec]
    estimator = spec["estimator"]
    include_fe = spec["include_fe"]
    _apply_preset_covariates(spec["covariates"])
    control_mode = "Preset mode: estimator, FE, and covariates are auto-applied from preset_spec."
else:
    control_mode = "Custom mode: estimator, FE, and covariates follow your manual control choices."


def get_selected_covariates() -> list[str]:
    return [
        covariate
        for covariate, var_name in COVARIATE_CHECKBOX_VARS.items()
        if bool(globals().get(var_name, False))
    ]


selected_covariates = get_selected_covariates()
if not selected_covariates:
    print("- warning: no covariate selected yet. In Custom mode, tick at least one checkbox before running the regression.")

print("Applied configuration")
print("- preset_spec:", preset_spec)
print("- estimator:", estimator)
print("- include_fe:", include_fe)
print("- selected_covariates:", ", ".join(selected_covariates))
print("- exporters filter:", exporters)
print("- importers filter:", importers)
print("- mode:", control_mode)
print("- note: comcur is not available in Gravity_V202010.dta and is not offered here.")

In [ ]:
# @title Run Gravity Regression

# If Colab misses any package, uncomment the next line.
# !pip -q install statsmodels

import io
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

try:
    from IPython.display import display, Markdown
except Exception:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

DATA_URL = "https://raw.githubusercontent.com/DTEcon/Teaching_International_Trade_PUBLIC/main/data/gravity_2019_student.csv"


def load_student_data() -> pd.DataFrame:
    try:
        data = pd.read_csv(DATA_URL)
        print(f"Loaded data from URL: {DATA_URL}")
        return data
    except Exception as exc:
        print("Could not load dataset from URL.")
        print(f"Reason: {exc}")
        print("Please upload gravity_2019_student.csv from LMS/repo.")
        try:
            from google.colab import files  # type: ignore
            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("No file uploaded.")
            first_name = next(iter(uploaded))
            data = pd.read_csv(io.BytesIO(uploaded[first_name]))
            print(f"Loaded uploaded file: {first_name}")
            return data
        except Exception as upload_exc:
            raise RuntimeError(
                "Data load failed from both URL and upload path."
            ) from upload_exc


df_full = load_student_data()
print(f"Rows: {len(df_full):,} | Columns: {len(df_full.columns)}")

EXPORTER_ONLY = {
    "gdp_o",
    "ln_gdp_o",
    "pop_o",
    "ln_pop_o",
    "gdpcap_o",
    "ln_gdpcap_o",
    "wto_o",
    "eu_o",
}
IMPORTER_ONLY = {
    "gdp_d",
    "ln_gdp_d",
    "pop_d",
    "ln_pop_d",
    "gdpcap_d",
    "ln_gdpcap_d",
    "wto_d",
    "eu_d",
}


def parse_iso_list(value: str) -> list[str]:
    value = str(value).strip()
    if value.upper() == "ALL" or value == "":
        return []
    return [item.strip().upper() for item in value.split(",") if item.strip()]


def stars(pvalue: float) -> str:
    if pd.isna(pvalue):
        return ""
    if pvalue < 0.01:
        return "***"
    if pvalue < 0.05:
        return "**"
    if pvalue < 0.1:
        return "*"
    return ""


def apply_fe_absorption_rules(covariate_list: list[str], include_fe: bool) -> tuple[list[str], list[tuple[str, str]]]:
    if not include_fe:
        return list(covariate_list), []

    kept = []
    dropped = []
    for var in covariate_list:
        if var in EXPORTER_ONLY:
            dropped.append((var, "exporter FE"))
        elif var in IMPORTER_ONLY:
            dropped.append((var, "importer FE"))
        else:
            kept.append(var)
    return kept, dropped


def drop_no_variation_covariates(
    data: pd.DataFrame,
    covariate_list: list[str],
) -> tuple[list[str], list[tuple[str, str]]]:
    kept = []
    dropped = []

    for var in covariate_list:
        nunique = data[var].nunique(dropna=True)
        if pd.isna(nunique) or nunique <= 1:
            dropped.append((var, "no variation in estimation sample"))
        else:
            kept.append(var)

    return kept, dropped


def drop_log_identity_covariates(
    covariate_list: list[str],
) -> tuple[list[str], list[tuple[str, str]]]:
    kept = list(covariate_list)
    dropped = []

    rules = [
        (["ln_gdp_o", "ln_pop_o", "ln_gdpcap_o"], "ln_gdpcap_o", "log identity ln_gdp_o = ln_pop_o + ln_gdpcap_o + constant"),
        (["ln_gdp_d", "ln_pop_d", "ln_gdpcap_d"], "ln_gdpcap_d", "log identity ln_gdp_d = ln_pop_d + ln_gdpcap_d + constant"),
    ]

    for required, drop_var, reason in rules:
        if all(var in kept for var in required) and drop_var in kept:
            kept.remove(drop_var)
            dropped.append((drop_var, reason))

    return kept, dropped


def drop_exact_linear_collinearity(
    data: pd.DataFrame,
    covariate_list: list[str],
    tolerance: float = 1e-4,
) -> tuple[list[str], list[tuple[str, str]]]:
    if not covariate_list:
        return [], []

    kept = []
    dropped = []
    base_matrix = np.ones((len(data), 1), dtype=float)

    for var in covariate_list:
        x = data[var].to_numpy(dtype=float).reshape(-1, 1)
        beta, *_ = np.linalg.lstsq(base_matrix, x, rcond=None)
        resid = x - base_matrix @ beta
        rel_resid = np.linalg.norm(resid) / max(np.linalg.norm(x), 1.0)

        if rel_resid <= tolerance:
            dropped.append((var, f"near-linear collinearity with other selected covariates (tol={tolerance:g})"))
        else:
            kept.append(var)
            base_matrix = np.column_stack([base_matrix, x])

    return kept, dropped


def build_formula(dep_var: str, covariate_list: list[str], include_fe: bool) -> str:
    rhs_terms = list(covariate_list)
    if include_fe:
        rhs_terms.append("C(iso3_o)")
        rhs_terms.append("C(iso3_d)")
    if not rhs_terms:
        rhs_terms = ["1"]
    return dep_var + " ~ " + " + ".join(rhs_terms)


def fit_gravity_model(
    data: pd.DataFrame,
    estimator_name: str,
    covariate_list: list[str],
    include_fe: bool,
    exporter_filter: list[str],
    importer_filter: list[str],
):
    work = data.copy()

    if exporter_filter:
        work = work[work["iso3_o"].isin(exporter_filter)].copy()
    if importer_filter:
        work = work[work["iso3_d"].isin(importer_filter)].copy()

    required = ["iso3_o", "iso3_d", "pair_cluster", "trade_value"] + covariate_list
    missing = [col for col in required if col not in work.columns]
    if missing:
        raise ValueError(f"Missing columns in data: {missing}")

    if estimator_name == "OLS":
        work = work[work["trade_value"] > 0].copy()
        work["ln_trade"] = np.log(work["trade_value"])
        dep_var = "ln_trade"
        fit_data = work.dropna(subset=[dep_var] + covariate_list + ["pair_cluster"])
    elif estimator_name == "PPML":
        dep_var = "trade_value"
        fit_data = work.dropna(subset=covariate_list + ["pair_cluster", "trade_value"])
    else:
        raise ValueError("Estimator must be OLS or PPML")

    if fit_data.empty:
        raise ValueError("No rows left after filters and NA handling.")

    model_covariates, sample_dropped_covariates = drop_no_variation_covariates(fit_data, covariate_list)
    model_covariates, identity_dropped_covariates = drop_log_identity_covariates(model_covariates)
    model_covariates, exact_collinear_dropped_covariates = drop_exact_linear_collinearity(fit_data, model_covariates)

    formula = build_formula(dep_var, model_covariates, include_fe)

    if estimator_name == "OLS":
        result = smf.ols(formula, data=fit_data).fit(
            cov_type="cluster",
            cov_kwds={"groups": fit_data["pair_cluster"]},
        )
    else:
        result = smf.glm(
            formula,
            data=fit_data,
            family=sm.families.Poisson(),
        ).fit(
            maxiter=200,
            cov_type="cluster",
            cov_kwds={"groups": fit_data["pair_cluster"]},
        )

    return (
        result,
        fit_data,
        formula,
        model_covariates,
        sample_dropped_covariates,
        identity_dropped_covariates,
        exact_collinear_dropped_covariates,
    )


def build_result_table(result, covariate_list: list[str]) -> pd.DataFrame:
    rows = []
    for var in covariate_list:
        coef = result.params.get(var, np.nan)
        se = result.bse.get(var, np.nan)
        pval = result.pvalues.get(var, np.nan)
        rows.append(
            {
                "variable": var,
                "coef": coef,
                "std_err": se,
                "p_value": pval,
                "stars": stars(pval),
            }
        )

    out = pd.DataFrame(rows, columns=["variable", "coef", "std_err", "p_value", "stars"])
    if out.empty:
        out["coef_with_stars"] = pd.Series(dtype="object")
        out["std_err_fmt"] = pd.Series(dtype="object")
        return out

    out["coef_with_stars"] = out.apply(
        lambda row: "" if pd.isna(row["coef"]) else f"{row['coef']:.4f}{row['stars']}",
        axis=1,
    )
    out["std_err_fmt"] = out["std_err"].apply(lambda x: "" if pd.isna(x) else f"({x:.4f})")
    return out


def absorbed_diagnostics(
    result_table: pd.DataFrame,
    fit_data: pd.DataFrame,
    model_covariates: list[str],
    fe_dropped_covariates: list[tuple[str, str]],
    sample_dropped_covariates: list[tuple[str, str]],
    identity_dropped_covariates: list[tuple[str, str]],
    exact_collinear_dropped_covariates: list[tuple[str, str]],
) -> list[str]:
    notes = []

    for var, absorber in fe_dropped_covariates:
        notes.append(
            f"{var}: automatically excluded because it is absorbed by {absorber} when include_fe=True."
        )

    for var, reason in sample_dropped_covariates:
        notes.append(f"{var}: automatically excluded because of {reason}.")

    for var, reason in identity_dropped_covariates:
        notes.append(f"{var}: automatically excluded because of {reason}.")

    for var, reason in exact_collinear_dropped_covariates:
        notes.append(f"{var}: automatically excluded because of {reason}.")

    if "dist" in model_covariates and "ln_dist" in model_covariates:
        corr = fit_data[["dist", "ln_dist"]].corr().iloc[0, 1]
        notes.append(
            f"dist and ln_dist are not exactly collinear (both are estimable), but are highly correlated in this sample (corr={corr:.4f})."
        )

    dropped = result_table[result_table["coef"].isna()]
    for var in dropped["variable"].tolist():
        notes.append(f"{var}: coefficient not estimated (dropped/collinear or no variation in sample).")

    if not notes:
        notes.append("No obvious FE-absorption/collinearity flags for the estimated covariates.")
    return notes


selected_covariates = get_selected_covariates()
if not selected_covariates:
    raise ValueError("Please select at least one covariate.")

covariates_after_fe, fe_dropped_covariates = apply_fe_absorption_rules(selected_covariates, include_fe)

selected_exporters = parse_iso_list(exporters)
selected_importers = parse_iso_list(importers)

(
    result,
    fit_data,
    formula,
    model_covariates,
    sample_dropped_covariates,
    identity_dropped_covariates,
    exact_collinear_dropped_covariates,
) = fit_gravity_model(
    data=df_full,
    estimator_name=estimator,
    covariate_list=covariates_after_fe,
    include_fe=include_fe,
    exporter_filter=selected_exporters,
    importer_filter=selected_importers,
)

summary_table = build_result_table(result, model_covariates)
notes = absorbed_diagnostics(
    summary_table,
    fit_data,
    model_covariates,
    fe_dropped_covariates,
    sample_dropped_covariates,
    identity_dropped_covariates,
    exact_collinear_dropped_covariates,
)

print("Run configuration")
print("- mode:", control_mode)
print("- requested_covariates:", ", ".join(selected_covariates))
print(
    "- estimated_covariates:",
    ", ".join(model_covariates) if model_covariates else "(none; FE-only specification)",
)
if fe_dropped_covariates:
    print("- auto_dropped_with_fe:", ", ".join(var for var, _ in fe_dropped_covariates))
if sample_dropped_covariates:
    print("- auto_dropped_no_variation:", ", ".join(var for var, _ in sample_dropped_covariates))
if identity_dropped_covariates:
    print("- auto_dropped_log_identity:", ", ".join(var for var, _ in identity_dropped_covariates))
if exact_collinear_dropped_covariates:
    print("- auto_dropped_exact_collinearity:", ", ".join(var for var, _ in exact_collinear_dropped_covariates))
print()
print("Model formula:")
print(formula)
print()
print("Sample diagnostics:")
print(f"- N (estimation sample): {len(fit_data):,}")
print(f"- Number of exporters in sample: {fit_data['iso3_o'].nunique():,}")
print(f"- Number of importers in sample: {fit_data['iso3_d'].nunique():,}")
if estimator == "PPML":
    zero_share = (fit_data["trade_value"] == 0).mean()
    print(f"- Zero-flow share used by PPML: {100 * zero_share:.2f}%")

print()
print("Coefficient table (clustered SE by pair):")
if summary_table.empty:
    print("No slope covariates estimated in this specification.")
else:
    display(summary_table[["variable", "coef_with_stars", "std_err_fmt", "p_value"]])

print("FE / collinearity diagnostics:")
for note in notes:
    print("-", note)

print()
print("Interpretation reminder (see Lecture 7):")
print("- Log variables (e.g., ln_dist): the coefficient is an elasticity.")
print("  A 1% increase in the variable changes trade by beta%.")
print("- Dummy variables (e.g., contig, comlang_off, rta): approximate effect is 100*kappa%.")
print("  Exact multiplicative effect: exp(kappa). Exact percent change: 100*(exp(kappa) - 1)%.")
print("  Example: kappa = 0.50 -> approx. 50%; exact = 100*(exp(0.50) - 1) = 64.9%.")
